In [ ]:
import numpy as np
import matplotlib.pyplot as plt 
import pandas as pd 
import os

from ripser import ripser
from persim import plot_diagrams
from gtda.diagrams import PersistenceEntropy, PairwiseDistance
from gtda.diagrams import Silhouette
from gtda.diagrams import BettiCurve 
from gtda.time_series import TakensEmbedding, SlidingWindow, SingleTakensEmbedding, takens_embedding_optimal_parameters
import plotly.express as px
from gtda.homology import VietorisRipsPersistence
from gtda.metaestimators import CollectionTransformer
from gtda.pipeline import Pipeline
from sklearn.decomposition import PCA
from gtda.plotting import plot_point_cloud

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

from tda_slugging.utils import read_files, find_shortest_file, batch_analyzer, mutual_information

# Data paths (type-3 slugging is committed; other types need the full 3W dataset)
DATA_3W_ALL      = str(REPO_ROOT / 'data' / '3W' / 'ALL')
DATA_3W_REAL     = str(REPO_ROOT / 'data' / '3W' / 'REAL')
DATA_3W_SIM      = str(REPO_ROOT / 'data' / '3W' / 'SIMULATED')
DATA_WELL        = str(REPO_ROOT / 'data' / 'well')
OUTPUTS_DIR      = str(REPO_ROOT / 'outputs')
IMAGES_DIR       = str(REPO_ROOT / 'images')
# Paths below require the full 3W dataset (download from https://github.com/ricardovvargas/3w_dataset):
# DATA_3W_NORMAL   = '/path/to/3w_dataset/data/0'
# DATA_3W_UNSTABLE = '/path/to/3w_dataset/data/4'


In [ ]:
x_timestamp = np.linspace(0, 10, 1000)
y_periodic = np.cos(x_timestamp)

n = 3
m = 2
y_N_periodic = np.cos(n * x_timestamp)
y_scaled = m * y_periodic 
y_shifted = m*n + y_periodic

In [ ]:
figShapes = px.line()
figShapes.add_scatter(x=(x_timestamp), y=(y_periodic), name="cos(x)")
#figShapes.add_scatter(x=(x_timestamp), y=(y_N_periodic), name="cos(n*x)")
figShapes.add_scatter(x=(x_timestamp), y=(y_scaled), name="m*cos(x)")
figShapes.show()

In [ ]:
stride = 1
signal = y_periodic

embedding_dimension = 3
embedding_time_delay = 10 #optimal_time_delay
embedder = SingleTakensEmbedding(
    parameters_type="fixed", n_jobs=-1, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

signal_embedded_cosx = embedder.fit_transform(signal)
fig = plot_point_cloud(signal_embedded_cosx)
fig

In [ ]:
def is_pos_def(x):
    return np.all(np.linalg.eigvals(x) >= 0)

print(signal_embedded_cosx.shape)
tmp = signal_embedded_cosx 
is_pos_def(signal_embedded_cosx[:3,:])

In [ ]:
np.linalg.eigvals(signal_embedded_cosx[:3,:])

In [ ]:
if not os.path.exists("images"):
    os.mkdir("images")
fig.write_image("images/fig1.svg")

In [ ]:
homology_dimensions = (0,1)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)
PerDiagram_cosx = VRP.fit_transform_plot(signal_embedded_cosx[None,:,:])

In [ ]:
file1 = open("cosx_persdiag_H0.dat", "w")
file2 = open("cosx_persdiag_H1.dat", "w")
for i in PerDiagram_cosx[0,:]:
    if i[2] == 0:
        tmp = str(i[0]) + str(' ') + str(i[1]) + "\n"
        file1.writelines(tmp)
    elif i[2] == 1:
        tmp = str(i[0]) + str(' ') + str(i[1]) + "\n"
        file2.writelines(tmp)
file1.close()
file2.close()

In [ ]:
Betti = BettiCurve()
Betti_cosx = Betti.fit_transform_plot(PerDiagram_cosx)
Betti_cosx

In [ ]:
from random import gauss
from random import seed
# seed random number generator
seed(123456)
# create white noise series
white_noise_gen = [gauss(0.0, .50) for i in range(1000)]
white_noise = pd.Series(white_noise_gen)

stride = 2
signal = white_noise

embedding_dimension = 3 
embedding_time_delay = 100 #optimal_time_delay
embedder = SingleTakensEmbedding(
    parameters_type="fixed", n_jobs=-1, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

signal_embedded_wn = embedder.fit_transform(signal)
fig_noise = plot_point_cloud(signal_embedded_wn)
fig_noise

In [ ]:
PerDiagram_cosx = VRP.fit_transform_plot(signal_embedded_cosx[None,:,:])

In [ ]:
file1 = open("wn_persdiag_H0.dat", "w")
file2 = open("wn_persdiag_H1.dat", "w")
for i in PerDiagram_cosx[0,:]:
    if i[2] == 0:
        tmp = str(i[0]) + str(' ') + str(i[1]) + "\n"
        file1.writelines(tmp)
    elif i[2] == 1:
        tmp = str(i[0]) + str(' ') + str(i[1]) + "\n"
        file2.writelines(tmp)
file1.close()
file2.close()

In [ ]:
stride = 1
signal = y_shifted

embedding_dimension = 3 
embedding_time_delay = 10 #optimal_time_delay
embedder = SingleTakensEmbedding(
    parameters_type="fixed", n_jobs=-1, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

signal_embedded_shift = embedder.fit_transform(signal)
fig = plot_point_cloud(signal_embedded_shift)
fig

In [ ]:
stride = 1
signal = y_N_periodic

embedding_dimension = 3 
embedding_time_delay = 10 #optimal_time_delay
embedder = SingleTakensEmbedding(
    parameters_type="fixed", n_jobs=-1, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

signal_embedded_cos3x = embedder.fit_transform(signal)
plot_point_cloud(signal_embedded_cos3x)

In [ ]:
signal = y_scaled

embedding_dimension = 3 
embedding_time_delay = 10 #optimal_time_delay
embedder = SingleTakensEmbedding(
    parameters_type="fixed", n_jobs=-1, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

signal_embedded_2cosx = embedder.fit_transform(signal)
plot_point_cloud(signal_embedded_2cosx)

In [ ]:
homology_dimensions = (0,1)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)
PerDiagram_cosx = VRP.fit_transform_plot(signal_embedded_cosx[None,:,:])

In [ ]:
PerDiagram_cosx

In [ ]:
PerDiagram_2cosx = VRP.fit_transform_plot(signal_embedded_2cosx[None,:,:])

In [ ]:
PerDiagram_cos3x = VRP.fit_transform_plot(signal_embedded_cos3x[None,:,:])

In [ ]:
Betti = BettiCurve(n_bins=100)
Betti_cosx = Betti.fit_transform_plot(PerDiagram_cosx)

In [ ]:
Betti.get_params()

In [ ]:
y_non_periodic_simple = np.cos(2*x_timestamp)*np.sin((x_timestamp*np.pi*1.5))

In [ ]:
from scipy import special
x = np.linspace(-25, 5, 501)
y_non_periodic, aip, bi, bip = special.airy(x)
y_non_periodic = y_non_periodic * np.sin(np.pi * x) * np.cos(2*x)

In [ ]:
figShapes = px.line()
figShapes.add_scatter(x=(x_timestamp), y=(y_non_periodic_simple), name="quasi-periodic")
figShapes.show()

In [ ]:
stride = 1
signal = y_non_periodic

from sklearn.decomposition import PCA

embedding_dimension = 5
embedding_time_delay = 25 #optimal_time_delay
embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=-1, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

signal_embedded_cosx = embedder.fit_transform(signal)
pca = PCA(n_components=3)
pca_slugging = pca.fit_transform(signal_embedded_cosx)

plot_point_cloud(pca_slugging)

In [ ]:
x = np.linspace(0, 10, 501)
alpha = 1 / np.log(5)
beta = 1 / np.log(2)
y_non_periodic = np.pi*np.sin(2* np.pi * alpha * x) - np.cos(2* np.pi * beta * x)

In [ ]:
figShapes = px.line()
figShapes.add_scatter(x=(x_timestamp), y=(y_non_periodic_simple), name="quasi-periodic")
figShapes.show()

In [ ]:
stride = 1
signal = y_non_periodic

from sklearn.decomposition import PCA

embedding_dimension = 8
embedding_time_delay = 25 #optimal_time_delay
embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=-1, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

signal_embedded_cosx = embedder.fit_transform(signal)
#print(embedding_dimension)
#if embedding_dimension > 3
pca = PCA(n_components=3)
pca_slugging = pca.fit_transform(signal_embedded_cosx)
#else:
  #  pca_slugging = signal_embedded_cosx
#pca_slugging = signal_embedded_cosx

plot_point_cloud(pca_slugging)

In [ ]:
if not os.path.exists("images"):
    os.mkdir("images")
fig.write_image("images/incomm.svg")

In [ ]:
PerDiagram = VRP.fit_transform_plot(signal_embedded_cosx[None,:,:])

## Non-commensurate two-frequency signal — torus topology

We study the signal

$$y(x) = \pi\sin(2\pi\alpha\, x) - \cos(2\pi\beta\, x), \qquad x \in [0, 10]$$

with

$$\alpha = \frac{1}{\ln 5} \approx 0.621, \qquad \beta = \frac{1}{\ln 2} \approx 1.443$$

The ratio $\alpha/\beta = \ln 2 / \ln 5 = \log_5 2$ is irrational (in fact transcendental), so the two frequencies are **genuinely incommensurate** — the signal is quasiperiodic and never repeats exactly.

### Why a torus and not a cylinder?

For a sliding-window (Takens) embedding of a *scalar* quasiperiodic signal to resolve the full **2-torus** $\mathbb{T}^2$ — rather than just a cylinder — two conditions must hold simultaneously:

1. **Both frequencies must be visibly present in the window.**  
   Here $T_1 = 1/\alpha \approx 1.61$ and $T_2 = 1/\beta \approx 0.69$ are well separated (ratio ≈ 2.3), so a window of moderate length captures roughly independent oscillations in both.

2. **The embedding dimension must be ≥ 4** (Whitney: $2 \times 2 = 4$).  
   With `parameters_type="search"` giotto-tda selects $d$ and $\tau$ via mutual-information / false-nearest-neighbours criteria; for this signal it typically finds $d \geq 4$.

When both conditions hold, the Vietoris–Rips persistence diagram should show:
- **two persistent $H_1$ bars** (the two independent loops of $\mathbb{T}^2$)
- **one persistent $H_2$ bar** (the enclosed 2-cavity)

In [ ]:
# ── Signal definition ─────────────────────────────────────────────────────
x = np.linspace(0, 10, 501)
alpha = 1 / np.log(5)   # ≈ 0.621
beta  = 1 / np.log(2)   # ≈ 1.443
y_qp  = np.pi * np.sin(2 * np.pi * alpha * x) - np.cos(2 * np.pi * beta * x)

print(f"α = {alpha:.4f}   T₁ = {1/alpha:.3f}")
print(f"β = {beta:.4f}   T₂ = {1/beta:.3f}")
print(f"α/β = {alpha/beta:.6f}  (irrational: log₅2 ≈ 0.431)")

# ── Signal plot ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 5))

axes[0].plot(x, np.pi * np.sin(2 * np.pi * alpha * x),
             lw=0.9, color='steelblue',  label=r'$\pi\sin(2\pi\alpha x)$')
axes[0].plot(x, -np.cos(2 * np.pi * beta * x),
             lw=0.9, color='darkorange', label=r'$-\cos(2\pi\beta x)$')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.2)
axes[0].set_title('Individual components', fontsize=10)

axes[1].plot(x, y_qp, lw=0.8, color='crimson',
             label=r'$y = \pi\sin(2\pi\alpha x) - \cos(2\pi\beta x)$')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.2)
axes[1].set_title('Quasiperiodic sum (never repeats exactly)', fontsize=10)
axes[1].set_xlabel('x')

plt.suptitle('Non-commensurate two-frequency signal  '
             r'($\alpha/\beta = \log_5 2$ — irrational)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Optimal Takens embedding via giotto-tda parameter search ──────────────
# parameters_type="search": τ from first minimum of mutual information,
# d from false-nearest-neighbours criterion (Kennel et al. 1992).
# Upper bounds: dimension=8, time_delay=40 (generous for both periods).
embedder_qp = SingleTakensEmbedding(
    parameters_type="search",
    time_delay=40,
    dimension=8,
    stride=1,
    n_jobs=-1,
)
signal_embedded_qp = embedder_qp.fit_transform(y_qp)

d_opt   = int(embedder_qp.dimension_)
tau_opt = int(embedder_qp.time_delay_)
print(f"Optimal embedding:  d = {d_opt},  τ = {tau_opt} samples")
print(f"Point cloud shape:  {signal_embedded_qp.shape}")
print(f"τ in x-units: {tau_opt * (x[1]-x[0]):.4f}   "
      f"≈ {tau_opt * (x[1]-x[0]) / (1/alpha):.2f} × T₁"
      f"  =  {tau_opt * (x[1]-x[0]) / (1/beta):.2f} × T₂")

In [ ]:
# ── PCA to 3D and point-cloud visualisation ───────────────────────────────
pca_qp = PCA(n_components=3).fit_transform(signal_embedded_qp)

fig_qp = plot_point_cloud(pca_qp)
fig_qp.update_layout(
    title=(f"Quasiperiodic signal — Takens embedding (d={d_opt}, τ={tau_opt}), "
           "PCA 3D<br><sup>Expect a donut / torus shape</sup>"),
    scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
)
fig_qp.show()

# ── Static 2D PCA projection for comparison ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
pairs = [(0, 1, "PC1", "PC2"), (0, 2, "PC1", "PC3"), (1, 2, "PC2", "PC3")]
for ax, (i, j, xi, yj) in zip(axes, pairs):
    ax.scatter(pca_qp[:, i], pca_qp[:, j],
               c=np.arange(len(pca_qp)), cmap="plasma", s=4, alpha=0.7)
    ax.set_xlabel(xi); ax.set_ylabel(yj)
    ax.set_aspect("equal"); ax.grid(alpha=0.2)
plt.suptitle("Quasiperiodic signal — PCA projections (filled annulus ≈ torus slice)", fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# ── Vietoris–Rips persistence with H0, H1, H2 ────────────────────────────
# We include H2 to look for the 2-cavity of the torus.
VRP_torus = VietorisRipsPersistence(homology_dimensions=(0, 1, 2), n_jobs=-1)
PerDiagram_qp = VRP_torus.fit_transform(signal_embedded_qp[None, :, :])

# ── Plot persistence diagram ───────────────────────────────────────────────
VRP_torus.fit_transform_plot(signal_embedded_qp[None, :, :])

# ── Summary: top bars per homology dimension ───────────────────────────────
print("\nTop persistent bars:")
for dim_val, name in [(0, "H₀"), (1, "H₁"), (2, "H₂")]:
    pts = PerDiagram_qp[0]
    mask = (pts[:, 2] == dim_val) & (pts[:, 1] > pts[:, 0])
    if not mask.any():
        print(f"  {name}: no finite bars")
        continue
    bars = pts[mask][:, :2]
    pers = bars[:, 1] - bars[:, 0]
    top  = np.argsort(pers)[::-1][:3]
    for rank, idx in enumerate(top, 1):
        b, d = bars[idx]
        print(f"  {name} bar {rank}: birth={b:.4f}  death={d:.4f}  pers={d-b:.4f}")

In [ ]:
# ── Betti curves — H0, H1, H2 over the filtration ────────────────────────
Betti_torus = BettiCurve(n_bins=100)
Betti_torus.fit_transform_plot(PerDiagram_qp)

# ── Manual ripser diagram for clean matplotlib comparison with simple signals ──
import ripser as _ripser
import persim as _persim

dgms = _ripser.ripser(signal_embedded_qp, maxdim=2)['dgms']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: persistence diagram
for k, (name, color) in enumerate([(r'$H_0$','steelblue'),
                                    (r'$H_1$','crimson'),
                                    (r'$H_2$','forestgreen')]):
    if k >= len(dgms): continue
    finite = dgms[k][np.isfinite(dgms[k][:, 1])]
    axes[0].scatter(finite[:, 0], finite[:, 1], c=color, s=20,
                    alpha=0.7, label=name)
dmax = max(d[np.isfinite(d[:,1]),1].max() for d in dgms if len(d) > 0 and np.isfinite(d[:,1]).any()) * 1.05
axes[0].plot([0, dmax], [0, dmax], 'k--', lw=0.8, alpha=0.4)
axes[0].set_xlabel('birth'); axes[0].set_ylabel('death')
axes[0].set_title('Persistence diagram', fontsize=10)
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.2)

# Right: barcode (H1 and H2 only — most informative)
offset = 0
for k, (name, color) in [(1,(r'$H_1$','crimson')), (2,(r'$H_2$','forestgreen'))]:
    if k >= len(dgms): continue
    finite = dgms[k][np.isfinite(dgms[k][:, 1])]
    pers   = finite[:, 1] - finite[:, 0]
    top    = np.argsort(pers)[::-1][:10]
    for rank, idx in enumerate(top):
        b, d = finite[idx]
        axes[1].barh(offset + rank, d - b, left=b,
                     color=color, alpha=0.8 if rank < 2 else 0.4, height=0.7)
    axes[1].axhline(offset - 0.5, color='gray', lw=0.6, ls='--')
    axes[1].text(-0.02, offset + len(top)/2 - 0.5, name,
                 va='center', ha='right', fontsize=9, color=color)
    offset += len(top) + 1

axes[1].set_xlabel('filtration value'); axes[1].set_yticks([])
axes[1].set_title('Barcode — top $H_1$ and $H_2$ bars', fontsize=10)
axes[1].grid(alpha=0.2)

plt.suptitle(r'Quasiperiodic signal — expect 2 persistent $H_1$ bars + 1 $H_2$ bar (torus $\mathbb{T}^2$)',
             fontsize=11)
plt.tight_layout(); plt.show()